# 05 · Função de Scoring

**Objetivo:** demonstrar `score_pessoa(...)` e `score_lote(...)` de `src/scoring.py`.

**Entradas:** `data/processed/rates/...`.  
**Saídas:** exemplos de score (em tela) e CSV de lote em `outputs/tables/`.

**Decisões:** binning idêntico ao do treino (sem *train/serve skew*);
EB aninhado + backoff resolvem células ausentes.  
**Limitações:** o score é a taxa da célula, não predição individual calibrada;
perfis idênticos recebem score idêntico.

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import load_config, anos_validos
cfg = load_config()
print('Raiz:', cfg['root'], '| modo sintético:', cfg['synthetic_mode'])

In [ ]:
from src import scoring
import json

# Pessoa exemplo (atributos brutos)
res = scoring.score_pessoa(
    cbo='252105', cnae='4711301', uf='PA', idade=33, escolaridade='superior',
    tempo_vinculo_meses=8, tamanho_empresa=2,
    motivos=['involuntario_sjc'], horizontes=[3, 6, 12])
print(json.dumps(res, ensure_ascii=False, indent=2, default=float))

In [ ]:
# Comparação: tempo de vínculo curto vs longo (espera-se risco maior p/ curto)
base = dict(cbo='521110', cnae='4711301', uf='PA', idade=40,
            escolaridade='medio_completo', tamanho_empresa=1)
for t in [2, 8, 18, 72]:
    r = scoring.score_pessoa(**base, tempo_vinculo_meses=t)
    r12 = r['risco']['involuntario_sjc'][12]
    print(f'tempo={t:>3}m -> risco 12m (sjc) = {r12:.3f} | nível={r["nivel_usado"]["involuntario_sjc"]}')

In [ ]:
# Score em lote
pessoas = pd.DataFrame([
    dict(cbo='252105', cnae='4711301', uf='PA', idade=33, escolaridade='superior',
         tempo_vinculo_meses=8, tamanho_empresa=2),
    dict(cbo='715210', cnae='4120400', uf='AM', idade=51, escolaridade='fundamental',
         tempo_vinculo_meses=40, tamanho_empresa=4),
    dict(cbo='621005', cnae='0111301', uf='TO', idade=27, escolaridade='medio_completo',
         tempo_vinculo_meses=3, tamanho_empresa=0),
])
scored = scoring.score_lote(pessoas, motivos=['involuntario_sjc','fim_contrato'])
scored.to_csv(cfg['abs']['tables'] / 'exemplo_score_lote.csv', index=False)
display(scored)